In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import joblib

In [ ]:
df = pd.read_csv("./loan_data1.csv")
df.head()

In [ ]:
df.info()

In [ ]:
print(df.isnull().mean()*100)

In [ ]:
df = df.drop(columns=['Loan_ID'],axis=1)

In [ ]:
df.head()

In [ ]:
df = df.dropna(subset = ['Gender', 'Dependents', 'Loan_Amount_Term'])

In [ ]:
df.isnull().sum()

In [ ]:

df['Self_Employed'].unique()

In [ ]:

df['Self_Employed'].value_counts()

In [ ]:

df['Self_Employed'].mode()[0]

In [ ]:

df['Credit_History'].unique()

In [ ]:

df['Credit_History'].mode()[0]

In [ ]:
# filling missing values with .mode()[0]
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace=True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0], inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df['Gender'].unique()

In [ ]:
df['Dependents'].unique()

In [ ]:
df['Dependents'].replace('3+', 4, inplace=True)
df['Dependents'].unique()

In [ ]:
# encoding our data
encoding = {
  'Gender': {'Male':1 , 'Female': 0}, 
  'Married': {'Yes': 1, 'No': 0},
  'Dependents': {'0':0, '1':1, '2': 2, '4': 4},
  'Education': {'Graduate': 1, 'Not Graduate': 0},
  'Self_Employed': {'Yes': 1, 'No': 0},
  'Property_Area': {'Rural': 0, 'Semiurban': 2, 'Urban': 1},
  'Loan_Status': {'Y': 1, 'N': 0}
}

In [ ]:
df.replace(encoding, inplace=True)

In [ ]:

X = df.drop('Loan_Status', axis = 1)
y = df['Loan_Status']

In [ ]:
df.head()

In [ ]:
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

In [ ]:
X.head()

In [ ]:
def evaluate_model(model):
  X_train, X_test, y_train, y_test  = train_test_split(X, y, test_size = 0.2, random_state = 42)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  cross_val = cross_val_score(model, X, y, cv=5)
  avg_cross_val = np.mean(cross_val)
  print(f"{model.__class__.__name__} - Accuarcy : {accuracy: .2f} , Cross-Val-Score: {avg_cross_val: .2f}")
  return avg_cross_val


In [ ]:
models = {
  LogisticRegression(),
  svm.SVC(),
  DecisionTreeClassifier(),
  RandomForestClassifier(),
  GradientBoostingClassifier(), 
}

In [ ]:
model_score = {model.__class__.__name__:evaluate_model(model) for model in models}

In [ ]:

def tune_model(model, param_grid):
  tuner = RandomizedSearchCV(model, param_grid, cv = 5, n_iter =20, verbose = True, random_state = 42)
  tuner.fit(X, y)
  print(f"Best Score for {model.__class__.__name__}: {tuner.best_score_:.2f}")
  print(f"Best Parameter for {model.__class__.__name__}: {tuner.best_params_}")
  return tuner.best_estimator_

In [ ]:

log_reg_grid = {'C': np.logspace(-4, 4, 20), "solver": ["liblinear"]}
svc_grid = {'C': [0.25, 0.50, 0.75, 1], "kernel": ['linear']}

rf_grid = {
  'n_estimators': np.arange(10, 1000, 10),
  'max_features': ['log2', 'sqrt'], 
  'max_depth': [None, 3, 5, 10, 20, 30],
  'min_samples_split': [2, 5, 20, 50, 100],
  'min_samples_leaf': [1, 2, 5, 10]
}

In [ ]:

best_log_reg = tune_model(LogisticRegression(), log_reg_grid)

In [ ]:
best_svc_reg = tune_model(svm.SVC(), svc_grid)

In [ ]:
best_rf = tune_model(RandomForestClassifier(), rf_grid)

In [ ]:
final_model = best_rf

In [ ]:
joblib.dump(final_model, 'loan_status_predictor.pkl')

In [ ]:
# Prediction System

sample_data = pd.DataFrame({
  'Gender': [1],
  'Married': [1],
  'Dependents': [2],
  'Education': [0],
  'Self_Employed': [0],
  'ApplicantIncome': [1000],
  'CoapplicantIncome': [0.0],
  'LoanAmount': [150],
  'Loan_Amount_Term': [180],
  'Credit_History': [0],
  'Property_Area': [1]
})

sample_data[num_cols] = scaler.transform(sample_data[num_cols])
loaded_model = joblib.load('loan_status_predictor.pkl')
prediction = loaded_model.predict(sample_data)

result = "Loan Approved" if prediction[0] == 1 else "Loan Not Approved"
print(f"\nPrediction Result: {result}")

In [ ]:
joblib.dump(scaler, 'vector.pkl')